# 64 Batch mesh statistics

Runs the anonymization pipeline on every valid thesis subject and collects the mesh-level quantities that Chapter 4 actually needs: vertex and face counts before and after deletion, the percentage of vertices removed, the cap-boundary height, and the degenerate-face percentage on the anonymized mesh.

The landmark-to-surface distance check that was here earlier was dropped: the five 10-20 landmarks are stored in the `_landmarks.tsv` sidecar and are therefore preserved by construction (the deletion operator does not touch the landmark array). Optode preservation is the science-relevant question and is handled in notebook 68. Face-detectability is handled in notebook 69.

Output: `thesis_results_out/batch_validation.csv`, which populates the mesh-statistics table and the mesh-integrity prose of Chapter 4.

**Prerequisite.** Each subject needs a `Subject{N}_anon_landmarks.tsv` sidecar, written by notebook 48. Subjects without the sidecar are skipped with a warning.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    SUBJECTS, subject_paths, load_raw, load_anon, load_landmarks,
    available_subjects, missing_report, run_pipeline,
)

import numpy as np
import pandas as pd

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

## 1. Which subjects are ready?

In [ ]:
ready = available_subjects()

print(f'Ready: {ready}')

missing = missing_report()

if missing:

    print('Missing files:')

    for n, items in missing.items():

        print(f'  Subject{n}: {items}')

## 2. Run pipeline per subject, collect mesh statistics

For each ready subject we load the raw scan and landmarks, run the pipeline, and record mesh statistics on the CTF-aligned head (`surface_ctf`) and the anonymized mesh (`surface_anon_ctf`). No validator call; just direct counts.

In [ ]:
rows = []

for n in ready:

    print(f'--- Subject{n} ---')

    surface_raw = load_raw(n)

    landmarks_raw = load_landmarks(n)

    art = run_pipeline(surface_raw, landmarks_raw, subject=n)



    n_head = art.surface_ctf.nvertices

    n_faces_head = art.surface_ctf.nfaces

    n_anon = art.surface_anon_ctf.nvertices

    n_faces_anon = art.surface_anon_ctf.nfaces

    mask_size = int(art.mask.sum())

    degen_area = (art.surface_anon_ctf.mesh.area_faces < 1e-12)

    degen_pct = 100.0 * float(degen_area.sum()) / max(1, len(degen_area))

    pct_removed = 100.0 * (n_head - n_anon) / max(1, n_head)



    row = {

        'subject': n,

        'n_vertices_raw': surface_raw.nvertices,

        'n_faces_raw': surface_raw.nfaces,

        'n_vertices_head': n_head,

        'n_faces_head': n_faces_head,

        'mask_size': mask_size,

        'n_vertices_anonymized': n_anon,

        'n_faces_anonymized': n_faces_anon,

        'vertices_removed': int(n_head - n_anon),

        'faces_removed': int(n_faces_head - n_faces_anon),

        'pct_vertices_removed': pct_removed,

        'degenerate_face_pct': degen_pct,

        'cap_z_mm': art.cap_z,

    }

    rows.append(row)

    print(

        f'  head: {n_head:,} v / {n_faces_head:,} f  ->  '

        f'anon: {n_anon:,} v / {n_faces_anon:,} f  '

        f'(-{pct_removed:.1f}%, degen {degen_pct:.3f}%, '

        f'cap_z {art.cap_z:.1f} mm)'

    )

## 3. Summary table

One row per subject. Columns feed the mesh-statistics table and the mesh-integrity prose of Chapter 4.

In [ ]:
df = pd.DataFrame(rows)

if len(df):

    df = df.sort_values('subject').reset_index(drop=True)

df

## 4. Save CSV

In [ ]:
out = OUT_DIR / 'batch_validation.csv'

df.to_csv(out, index=False)

print(f'Wrote {out} ({len(df)} rows)')